# Item-Based Collaborative Filtering on MovieLens 100k

This notebook walks through the same pipeline as `run_analysis.py`, importing the real implementation from `analysis.collaborative_filtering`, `analysis.evaluation`, and `data.load_movielens` (no duplicated logic). It:

1. Loads the real MovieLens 100k official train/test split (`ua.base` / `ua.test`)
2. Builds the user-item ratings matrix from the training set
3. Computes item-item cosine similarity
4. Evaluates RMSE and Precision@5 on the held-out test set
5. Prints example top-5 recommendations, with real movie titles, for three real users

In [1]:
from __future__ import annotations

import sys
from pathlib import Path

import numpy as np

# Allow imports from the project root when running from notebooks/
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from analysis.collaborative_filtering import (
    build_user_item_matrix,
    compute_item_similarity,
    predict_rating,
    recommend_for_user,
)
from analysis.evaluation import precision_at_k, rmse
from data.load_movielens import load_movies, load_train_test_split

K_NEIGHBORS = 20
TOP_N = 5

## 1. Load the real train/test split

`load_train_test_split()` reads MovieLens 100k's official `ua.base` (train) and `ua.test` (test) files -- 80,000 and 20,000 real ratings respectively.

In [2]:
train, test = load_train_test_split()
movies = load_movies().set_index("item_id")["title"]

print(f"Train: {len(train):,} real ratings. Test: {len(test):,} real ratings.")
train.head()

Train: 80,000 real ratings. Test: 20,000 real ratings.


,user_id,item_id,rating,timestamp
0,1,1,5,874965758
1,1,2,3,876893171
2,1,3,4,878542960
3,1,4,3,876893119
4,1,5,3,889751712


## 2. Build the user-item matrix and item-item similarity

The matrix is built from the training ratings only (no test leakage). Item-item cosine similarity is then computed from that matrix.

In [3]:
matrix = build_user_item_matrix(train)
similarity = compute_item_similarity(matrix)

print(f"User-item matrix shape: {matrix.shape}")
print(f"Item-item similarity shape: {similarity.shape}")

User-item matrix shape: (943, 1650)
Item-item similarity shape: (1650, 1650)


## 3. Evaluate RMSE on the held-out test set

For every test rating whose user and item are known to the training matrix, predict a rating with `predict_rating` and compare it against the real held-out rating with RMSE.

In [4]:
predictions, actuals = [], []
for row in test.itertuples():
    if row.user_id not in matrix.index or row.item_id not in matrix.columns:
        continue
    pred = predict_rating(matrix, similarity, row.user_id, row.item_id, k=K_NEIGHBORS)
    if not np.isnan(pred):
        predictions.append(pred)
        actuals.append(row.rating)

score = rmse(predictions, actuals)
print(f"RMSE on {len(predictions):,} real held-out test ratings: {score:.4f}")

RMSE on 19,968 real held-out test ratings: 1.0424


## 4. Evaluate Precision@5 across a sample of real users

For each of the first 50 real users in the test set who have at least one "relevant" held-out item (rating >= 4), generate the top-5 recommendations and score them with Precision@5.

In [5]:
precisions = []
sample_users = test["user_id"].unique()[:50]
for user_id in sample_users:
    if user_id not in matrix.index:
        continue
    relevant = set(test[(test["user_id"] == user_id) & (test["rating"] >= 4)]["item_id"])
    if not relevant:
        continue
    recs = recommend_for_user(matrix, similarity, user_id, n=TOP_N, k=K_NEIGHBORS)
    precisions.append(precision_at_k(recs, relevant, TOP_N))

mean_precision = float(np.mean(precisions)) if precisions else float("nan")
print(
    f"Mean Precision@{TOP_N} over {len(precisions)} real users with "
    f"relevant held-out items: {mean_precision:.4f}"
)

Mean Precision@5 over 50 real users with relevant held-out items: 0.0320


## 5. Example recommendations

Top-5 recommendations, with real movie titles, for three real users from the test set.

In [6]:
print("Example recommendations:")
for user_id in sample_users[:3]:
    if user_id not in matrix.index:
        continue
    recs = recommend_for_user(matrix, similarity, user_id, n=TOP_N, k=K_NEIGHBORS)
    titles = [movies.get(i, f"item {i}") for i in recs]
    print(f"  User {user_id}: {titles}")

Example recommendations:


  User 1: ['Convent, The (Convento, O) (1995)', 'Coldblooded (1995)', 'Farewell My Concubine (1993)', 'Wild Reeds (1994)', 'Brassed Off (1996)']


  User 2: ['Hearts and Minds (1996)', 'Coldblooded (1995)', 'Mamma Roma (1962)', 'Outlaw, The (1943)', "Jupiter's Wife (1994)"]


  User 3: ['Fille seule, La (A Single Girl) (1995)', 'Girl in the Cadillac (1995)', 'Mostro, Il (1994)', "Enfer, L' (1994)", 'Grosse Fatigue (1994)']
